# 📋 規格書：校園咖啡廳預約取餐系統

學號：41132579  姓名:蔡淳彥

## 1. 系統目的（一句話）

建立一套校園咖啡廳預約取餐系統，讓學生透過自然語言訂購飲品、取消訂單及查詢庫存，並讓店長查詢每日銷售統計。

## 2. 使用者會怎麼說（至少 5 句）

1. 我要兩杯拿鐵。
2. 幫我訂購一杯美式咖啡。
3. 我要取消上一筆拿鐵訂單。
4. 請問抹茶歐蕾還剩幾杯？
5. 我要三杯抹茶歐蕾。
6. 幫我查詢美式咖啡的庫存。
7. 今天各品項賣得怎麼樣？
8. 我要四杯拿鐵。


## 3. 資料表（2 張）

| 表名 | 欄位 | 型態 | 說明 |
|---|---|---|---|
| menu | item | TEXT PRIMARY KEY | 飲品名稱 |
| menu | price | INTEGER | 飲品單價 |
| menu | stock | INTEGER | 今日剩餘庫存 |
| orders | id | INTEGER PRIMARY KEY AUTOINCREMENT | 訂單編號 |
| orders | student_name | TEXT | 學生姓名 |
| orders | item | TEXT | 飲品名稱 |
| orders | quantity | INTEGER | 訂購數量 |
| orders | status | TEXT | 訂單狀態 |
| orders | created_at | TEXT | 訂單建立時間 |

**初始資料：**

| 品項 | 單價 | 今日庫存 |
|---|---:|---:|
| 美式咖啡 | 50 元 | 5 杯 |
| 拿鐵 | 65 元 | 4 杯 |
| 抹茶歐蕾 | 70 元 | 3 杯 |

每次執行程式時，先刪除舊資料表，再重新建立兩張資料表並放入上述初始資料。

訂單狀態分為「已成立」及「已取消」。

## 4. AI 的工作

- **輸入：** 使用者輸入的一句自然語言。
- **輸出格式：** `動作,品項,數量`
- **動作只能是：** 訂購、取消、查詢、統計、未知。
- **品項只能是：** 美式咖啡、拿鐵、抹茶歐蕾、未知。
- **數量的規則：**
  1. 從使用者句子辨識數量。
  2. 沒有指定數量時預設為 1。
  3. 數量必須為 1～3 杯。
  4. 超過允許範圍時由 Python 程式拒絕，不可建立訂單。
- **無法判斷時輸出：** 未知,未知,1，並提示使用者目前提供的飲品品項。
- **AI 不可以做的事：**
  1. 不可以自行修改資料庫。
  2. 不可以自行決定庫存是否足夠。
  3. 不可以自行計算或確認訂單金額。
  4. 不可以編造不存在的飲品。
  5. 不可以直接相信 AI 回傳的內容，必須由 Python 程式檢查允許的動作、品項及數量。

## 5. 功能與規則

| 功能 | 條件 | 系統回應 | 資料庫變化 |
|---|---|---|---|
| 訂購飲品 | 姓名及品項正確、庫存足夠、數量為 1～3 杯 | 顯示訂購成功及總金額 | 新增訂單並扣除庫存 |
| 訂購失敗 | 庫存不足 | 提示庫存不足 | 不修改資料 |
| 訂購失敗 | 數量不在 1～3 杯 | 提示數量超出限制 | 不修改資料 |
| 訂購失敗 | 當日訂購總數超過 3 杯 | 提示每人最多訂購 3 杯 | 不修改資料 |
| 取消訂單 | 姓名及品項正確，且有未取消訂單 | 顯示取消成功 | 將該學生該品項最新一筆有效訂單改為已取消，並釋出庫存 |
| 取消失敗 | 查無有效訂單 | 提示查無訂單 | 不修改資料 |
| 查詢庫存 | 品項存在 | 顯示剩餘庫存 | 不修改資料 |
| 查詢失敗 | 品項不存在 | 提示目前提供的品項 | 不修改資料 |
| 銷售統計 | 使用者要求查詢今日銷售 | 顯示各品項單價、庫存、已售杯數、剩餘杯數及營收 | 不修改資料 |

**補充規則：**

1. 訂購及取消必須輸入學生姓名，查詢及統計不需要姓名。
2. 每位學生今天最多訂購 3 杯，所有飲品數量合計計算。
3. 已取消的訂單不計入個人訂購上限及銷售數量。
4. 訂購金額＝飲品單價 × 訂購數量。
5. 統計必須使用 JOIN 與 GROUP BY。
6. 統計必須顯示所有飲品，包括沒有任何人訂購的飲品。

## 6. 例外情況（至少 3 個）

1. **庫存不足：** 當使用者訂購數量超過剩餘庫存時，系統拒絕訂購並提示目前剩餘杯數。
2. **找不到訂單：** 使用者取消不存在或已取消的訂單時，系統提示查無有效訂單。
3. **無法辨識飲品：** 使用者輸入不存在的飲品名稱時，系統提示目前提供的飲品品項。
4. **數量超出限制：** 使用者訂購數量小於 1 或大於 3 時，系統拒絕訂購。
5. **超過個人上限：** 學生當日有效訂購總數加上本次數量超過 3 杯時，系統拒絕訂購。
6. **AI 呼叫失敗：** Groq API 發生錯誤時，程式最多重試 3 次，仍失敗則提示使用者稍後再試。

## 7. 測試清單

### 7-1 AI 解讀測試（9 句）

| # | 類型 | 使用者說的話 | 預期動作 | 預期品項 | 預期數量 |
|---|---|---|---|---|---:|
| 1 | 正常 | 我要兩杯拿鐵 | 訂購 | 拿鐵 | 2 |
| 2 | 正常 | 幫我訂一杯美式咖啡 | 訂購 | 美式咖啡 | 1 |
| 3 | 正常 | 我要三杯抹茶歐蕾 | 訂購 | 抹茶歐蕾 | 3 |
| 4 | 正常 | 請問拿鐵還剩幾杯 | 查詢 | 拿鐵 | 1 |
| 5 | 正常 | 我要取消上一筆美式咖啡訂單 | 取消 | 美式咖啡 | 1 |
| 6 | 正常 | 今天各品項賣得怎麼樣 | 統計 | 未知 | 1 |
| 7 | 邊界 | 我要四杯拿鐵 | 訂購 | 拿鐵 | 4 |
| 8 | 錯誤 | 我要一杯珍珠奶茶 | 訂購 | 未知 | 1 |
| 9 | 錯誤 | 有哪些飲料可以買 | 未知 | 未知 | 1 |

In [12]:
# ============================================================
# 第 1 格：安裝套件、讀取金鑰、設定模型（老師提供，請勿修改）
# ============================================================
!pip install -q groq gradio

import os
import re
import time
import sqlite3
import pandas as pd
from IPython.display import display
from groq import Groq
import gradio as gr

# 判斷目前是在 Colab 還是本地端 Jupyter
try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 🔴 金鑰名稱一律為 GROQ_API_KEY；絕對不可以把金鑰寫在程式裡，也不可以 print 出來
if IN_COLAB:
    try:
        GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    except Exception:
        print("❌ 讀不到金鑰：請到左側 🔑 新增名稱為 GROQ_API_KEY 的密鑰，並打開「筆記本存取權」")
        raise
else:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
    if not GROQ_API_KEY:
        raise RuntimeError("找不到環境變數 GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

# 期中考指定模型；推理強度設為 low，可大幅減少 token 用量
# （唯一允許修改處：若 20b 額度用完，可改為 "openai/gpt-oss-120b"，並在規格書最後註明）
MODEL = "openai/gpt-oss-20b"
REASONING_EFFORT = "low"
print("✅ 環境：", "Colab" if IN_COLAB else "本地端", "｜模型：", MODEL)

✅ 環境： Colab ｜模型： openai/gpt-oss-20b


In [7]:
DB_PATH = "cafe.db"

def init_db():

    # 連接 SQLite 資料庫
    conn = sqlite3.connect(DB_PATH)

    try:
        cursor = conn.cursor()

        # 先刪除舊資料表
        cursor.execute("DROP TABLE IF EXISTS orders")
        cursor.execute("DROP TABLE IF EXISTS menu")

        # 建立菜單資料表
        cursor.execute("""
            CREATE TABLE menu (
                item TEXT PRIMARY KEY,
                price INTEGER NOT NULL,
                stock INTEGER NOT NULL
            )
        """)

        # 建立訂單資料表
        cursor.execute("""
            CREATE TABLE orders (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                student_name TEXT NOT NULL,
                item TEXT NOT NULL,
                quantity INTEGER NOT NULL,
                status TEXT NOT NULL,
                created_at TEXT NOT NULL,
                FOREIGN KEY (item) REFERENCES menu(item)
            )
        """)

        # 新增初始飲品資料
        menu_data = [
            ("美式咖啡", 50, 5),
            ("拿鐵", 65, 4),
            ("抹茶歐蕾", 70, 3)
        ]

        cursor.executemany("""
            INSERT INTO menu (item, price, stock)
            VALUES (?, ?, ?)
        """, menu_data)

        # 儲存資料
        conn.commit()

        print("✅ 資料庫建立成功！")
        print("✅ 菜單及初始資料新增成功！")

    except Exception as e:
        conn.rollback()
        print("❌ 資料庫建立失敗：", e)

    finally:
        # 確保資料庫連線關閉
        conn.close()


# 執行建立資料庫
init_db()

✅ 資料庫建立成功！
✅ 菜單及初始資料新增成功！


In [5]:
import sqlite3
import pandas as pd

DB_PATH = "cafe.db"

def show_tables():

    # 連接資料庫
    conn = sqlite3.connect(DB_PATH)

    try:
        # 顯示菜單資料表
        print("========== 菜單資料表 ==========")

        menu_df = pd.read_sql_query(
            "SELECT * FROM menu",
            conn
        )

        display(menu_df)

        # 顯示訂單資料表
        print("========== 訂單資料表 ==========")

        orders_df = pd.read_sql_query(
            "SELECT * FROM orders",
            conn
        )

        display(orders_df)

    except Exception as e:
        print("❌ 顯示資料表失敗：", e)

    finally:
        # 確保資料庫連線關閉
        conn.close()


# 呼叫函式，顯示一次資料表
show_tables()

========== 菜單資料表 ==========
❌ 顯示資料表失敗： Execution failed on sql 'SELECT * FROM menu': no such table: menu


In [13]:
# 暫存 AI 解讀結果，避免重複呼叫
ai_cache = {}

def ai_interpret(user_text):

    # 如果之前解讀過，直接使用暫存結果
    if user_text in ai_cache:
        return ai_cache[user_text]

    prompt = f"""
你是校園咖啡廳預約取餐系統的 AI 助手。

請分析使用者的句子：
{user_text}

請辨識以下三個欄位。

一、動作只能是：
訂購、取消、查詢、統計、未知

二、品項只能是：
美式咖啡、拿鐵、抹茶歐蕾、全部、未知

三、數量：
請辨識使用者指定的數量，並轉換成阿拉伯數字。
例如：
一杯 = 1
兩杯 = 2
三杯 = 3
四杯 = 4

沒有指定數量時預設為 1。

判斷規則：
1. 購買飲品為訂購。
2. 取消訂單為取消。
3. 詢問剩餘杯數為查詢。
4. 詢問今日各品項銷售情況為統計。
5. 統計時品項填寫全部。
6. 無法判斷時動作填寫未知。
7. 不存在的飲品品項填寫未知。
8. 不可以自行增加不存在的飲品。

只能輸出一行，格式如下：
動作,品項,數量

例如：
訂購,拿鐵,2

不要輸出其他解釋。
"""

    raw = ""

    # 最多呼叫 AI 三次
    for attempt in range(3):

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0,
                reasoning_effort=REASONING_EFFORT
            )

            raw = response.choices[0].message.content or ""

            # 每次呼叫後休息 2 秒
            time.sleep(2)

            break

        except Exception as e:

            error_text = str(e)

            # 判斷是否為 429 額度限制
            is_429 = (
                "429" in error_text
                or "rate_limit" in error_text.lower()
                or getattr(e, "status_code", None) == 429
            )

            if is_429 and attempt < 2:
                print("⚠️ AI 使用量限制，等待 30 秒後重試")
                time.sleep(30)

            else:
                raw = "未知,未知,1"
                break

    # 清理 AI 回覆
    raw = raw.strip()

    # 使用正規表示式檢查 AI 回覆格式
    pattern = (
        r"(訂購|取消|查詢|統計|未知)"
        r"\s*[,，]\s*"
        r"(美式咖啡|拿鐵|抹茶歐蕾|全部|未知)"
        r"\s*[,，]\s*"
        r"(\d+)"
    )

    match = re.search(pattern, raw)

    # 不符合格式就當作未知
    if match is None:

        result = ("未知", "未知", 1, raw)

    else:

        action = match.group(1)
        item = match.group(2)
        quantity = int(match.group(3))

        # 再次檢查允許的動作
        allowed_actions = [
            "訂購", "取消", "查詢", "統計", "未知"
        ]

        # 再次檢查允許的品項
        allowed_items = [
            "美式咖啡", "拿鐵", "抹茶歐蕾", "全部", "未知"
        ]

        if action not in allowed_actions:
            action = "未知"

        if item not in allowed_items:
            item = "未知"

        result = (action, item, quantity, raw)

    # 暫存結果
    ai_cache[user_text] = result

    return result


print("✅ AI 解讀函式建立完成")

✅ AI 解讀函式建立完成


In [14]:
# ============================================================
# 第 5 格：AI 解讀測試
# 提醒：8～10 句，要與規格書 7-1 一致
# ============================================================

# 測試資料
test_cases = [

    ("我要兩杯拿鐵", "訂購", "拿鐵", 2),

    ("幫我訂一杯美式咖啡", "訂購", "美式咖啡", 1),

    ("我要三杯抹茶歐蕾", "訂購", "抹茶歐蕾", 3),

    ("請問拿鐵還剩幾杯？", "查詢", "拿鐵", 1),

    ("幫我取消美式咖啡", "取消", "美式咖啡", 1),

    ("今天各品項賣得怎麼樣？", "統計", "全部", 1),

    ("我要四杯拿鐵", "訂購", "拿鐵", 4),

    ("我要一杯珍珠奶茶", "訂購", "未知", 1),

    ("今天天氣如何？", "未知", "未知", 1)

]

passed = 0

print("========== AI 解讀測試開始 ==========")

for i, test in enumerate(test_cases, 1):

    user_text, expected_action, expected_item, expected_qty = test

    # 呼叫 AI 解讀
    action, item, quantity, raw = ai_interpret(user_text)

    print(f"\n第 {i} 題")
    print("使用者輸入：", user_text)
    print("AI 原始回覆：", raw)

    print(
        "系統解讀：",
        action,
        item,
        quantity
    )

    print(
        "預期結果：",
        expected_action,
        expected_item,
        expected_qty
    )

    # 比對預期結果
    if (
        action == expected_action
        and item == expected_item
        and quantity == expected_qty
    ):

        print("✅ PASS")
        passed += 1

    else:

        print("❌ FAIL")

print("\n========== AI 測試結果 ==========")

print("總題數：", len(test_cases))
print("通過：", passed)
print("失敗：", len(test_cases) - passed)

print(
    "通過率：",
    round(passed / len(test_cases) * 100, 2),
    "%"
)

========== AI 解讀測試開始 ==========

第 1 題
使用者輸入： 我要兩杯拿鐵
AI 原始回覆： 未知,未知,1
系統解讀： 未知 未知 1
預期結果： 訂購 拿鐵 2
❌ FAIL

第 2 題
使用者輸入： 幫我訂一杯美式咖啡
AI 原始回覆： 未知,未知,1
系統解讀： 未知 未知 1
預期結果： 訂購 美式咖啡 1
❌ FAIL

第 3 題
使用者輸入： 我要三杯抹茶歐蕾
AI 原始回覆： 未知,未知,1
系統解讀： 未知 未知 1
預期結果： 訂購 抹茶歐蕾 3
❌ FAIL

第 4 題
使用者輸入： 請問拿鐵還剩幾杯？
AI 原始回覆： 未知,未知,1
系統解讀： 未知 未知 1
預期結果： 查詢 拿鐵 1
❌ FAIL

第 5 題
使用者輸入： 幫我取消美式咖啡
AI 原始回覆： 未知,未知,1
系統解讀： 未知 未知 1
預期結果： 取消 美式咖啡 1
❌ FAIL

第 6 題
使用者輸入： 今天各品項賣得怎麼樣？
AI 原始回覆： 未知,未知,1
系統解讀： 未知 未知 1
預期結果： 統計 全部 1
❌ FAIL

第 7 題
使用者輸入： 我要四杯拿鐵
AI 原始回覆： 未知,未知,1
系統解讀： 未知 未知 1
預期結果： 訂購 拿鐵 4
❌ FAIL

第 8 題
使用者輸入： 我要一杯珍珠奶茶
AI 原始回覆： 未知,未知,1
系統解讀： 未知 未知 1
預期結果： 訂購 未知 1
❌ FAIL

第 9 題
使用者輸入： 今天天氣如何？
AI 原始回覆： 未知,未知,1
系統解讀： 未知 未知 1
預期結果： 未知 未知 1
✅ PASS

========== AI 測試結果 ==========
總題數： 9
通過： 1
失敗： 8
通過率： 11.11 %


In [15]:
# ============================================================
# 第 6 格：功能函式
# 提醒：
# fetchone() 後先檢查 None
# 用 try/finally 關閉連線
# 統計使用 JOIN + GROUP BY
# ============================================================


# ============================================================
# 1. 查詢目前所有飲品
# ============================================================

def get_menu_items():

    conn = sqlite3.connect(DB_PATH)

    try:

        rows = conn.execute(
            "SELECT item FROM menu"
        ).fetchall()

        return [row[0] for row in rows]

    finally:
        conn.close()


# ============================================================
# 2. 訂購飲品
# ============================================================

def order_drink(student_name, item, quantity):

    # 檢查姓名
    if not student_name.strip():
        return "❌ 請輸入學生姓名。"

    # 檢查數量
    if quantity < 1 or quantity > 3:
        return "❌ 訂購數量必須在 1～3 杯之間。"

    conn = sqlite3.connect(DB_PATH)

    try:

        # 查詢飲品價格及庫存
        row = conn.execute(
            """
            SELECT price, stock
            FROM menu
            WHERE item = ?
            """,
            (item,)
        ).fetchone()

        # 檢查是否查得到
        if row is None:
            return "❌ 找不到此飲品。"

        price, stock = row

        # 查詢學生今日有效訂單總數
        today = time.strftime("%Y-%m-%d")

        row = conn.execute(
            """
            SELECT COALESCE(SUM(quantity), 0)
            FROM orders
            WHERE student_name = ?
            AND status = ?
            AND substr(created_at, 1, 10) = ?
            """,
            (student_name, "已成立", today)
        ).fetchone()

        if row is None:
            return "❌ 無法查詢訂單數量。"

        total_ordered = row[0]

        # 每人每日最多 3 杯
        if total_ordered + quantity > 3:

            return (
                "❌ 訂購失敗！\n"
                "每位學生每天最多訂購 3 杯。\n"
                f"目前已訂購：{total_ordered} 杯"
            )

        # 檢查庫存
        if stock < quantity:

            return (
                "❌ 庫存不足！\n"
                f"目前只剩 {stock} 杯。"
            )

        # 計算總金額
        total_price = price * quantity

        # 扣除庫存
        conn.execute(
            """
            UPDATE menu
            SET stock = stock - ?
            WHERE item = ?
            """,
            (quantity, item)
        )

        # 新增訂單
        created_at = time.strftime("%Y-%m-%d %H:%M:%S")

        conn.execute(
            """
            INSERT INTO orders
            (student_name, item, quantity, status, created_at)
            VALUES (?, ?, ?, ?, ?)
            """,
            (
                student_name,
                item,
                quantity,
                "已成立",
                created_at
            )
        )

        # 儲存資料
        conn.commit()

        return (
            "✅ 訂購成功！\n"
            f"學生姓名：{student_name}\n"
            f"飲品：{item}\n"
            f"數量：{quantity} 杯\n"
            f"單價：{price} 元\n"
            f"總金額：{total_price} 元\n"
            f"剩餘庫存：{stock - quantity} 杯"
        )

    except Exception as e:

        conn.rollback()

        return f"❌ 訂購發生錯誤：{e}"

    finally:
        conn.close()


# ============================================================
# 3. 取消訂單
# ============================================================

def cancel_order(student_name, item):

    if not student_name.strip():
        return "❌ 請輸入學生姓名。"

    conn = sqlite3.connect(DB_PATH)

    try:

        # 找出最新一筆有效訂單
        row = conn.execute(
            """
            SELECT id, quantity
            FROM orders
            WHERE student_name = ?
            AND item = ?
            AND status = ?
            ORDER BY id DESC
            LIMIT 1
            """,
            (student_name, item, "已成立")
        ).fetchone()

        # 檢查是否存在訂單
        if row is None:
            return "❌ 查無有效訂單，無法取消。"

        order_id, quantity = row

        # 檢查飲品是否存在
        menu_row = conn.execute(
            """
            SELECT stock
            FROM menu
            WHERE item = ?
            """,
            (item,)
        ).fetchone()

        if menu_row is None:
            return "❌ 找不到此飲品。"

        # 修改訂單狀態
        conn.execute(
            """
            UPDATE orders
            SET status = ?
            WHERE id = ?
            """,
            ("已取消", order_id)
        )

        # 釋出庫存
        conn.execute(
            """
            UPDATE menu
            SET stock = stock + ?
            WHERE item = ?
            """,
            (quantity, item)
        )

        conn.commit()

        return (
            "✅ 取消成功！\n"
            f"學生姓名：{student_name}\n"
            f"飲品：{item}\n"
            f"取消數量：{quantity} 杯\n"
            "庫存已恢復。"
        )

    except Exception as e:

        conn.rollback()

        return f"❌ 取消發生錯誤：{e}"

    finally:
        conn.close()


# ============================================================
# 4. 查詢剩餘庫存
# ============================================================

def query_stock(item):

    conn = sqlite3.connect(DB_PATH)

    try:

        row = conn.execute(
            """
            SELECT price, stock
            FROM menu
            WHERE item = ?
            """,
            (item,)
        ).fetchone()

        if row is None:
            return "❌ 找不到此飲品。"

        price, stock = row

        return (
            "📋 飲品庫存查詢\n"
            f"品項：{item}\n"
            f"單價：{price} 元\n"
            f"剩餘庫存：{stock} 杯"
        )

    except Exception as e:

        return f"❌ 查詢發生錯誤：{e}"

    finally:
        conn.close()


# ============================================================
# 5. 今日銷售統計
# 使用 LEFT JOIN + GROUP BY
# ============================================================

def sales_statistics():

    conn = sqlite3.connect(DB_PATH)

    try:

        today = time.strftime("%Y-%m-%d")

        sql = """
            SELECT
                m.item AS 品項,

                m.price AS 單價,

                (
                    m.stock +
                    COALESCE(SUM(o.quantity), 0)
                ) AS 庫存,

                COALESCE(
                    SUM(o.quantity), 0
                ) AS 已售杯數,

                m.stock AS 剩餘杯數,

                COALESCE(
                    SUM(o.quantity * m.price), 0
                ) AS 營收

            FROM menu AS m

            LEFT JOIN orders AS o
            ON m.item = o.item
            AND o.status = '已成立'
            AND substr(o.created_at, 1, 10) = ?

            GROUP BY
                m.item,
                m.price,
                m.stock

            ORDER BY m.item
        """

        df = pd.read_sql_query(
            sql,
            conn,
            params=(today,)
        )

        return (
            "========== 今日銷售統計 ==========\n\n"
            + df.to_string(index=False)
        )

    except Exception as e:

        return f"❌ 統計查詢發生錯誤：{e}"

    finally:
        conn.close()


print("✅ 訂購、取消、查詢及統計功能建立完成")

✅ 訂購、取消、查詢及統計功能建立完成


In [17]:
# ============================================================
# 第 7 格：主流程與流程測試
# 提醒：流程測試 8～10 步；最後呼叫 show_tables()
# ============================================================


# ============================================================
# 主流程
# ============================================================

def process_request(user_text, student_name=""):

    # 檢查使用者輸入
    if not user_text.strip():
        return "❌ 請輸入您的需求。"

    # AI 解讀
    action, item, quantity, raw = ai_interpret(user_text)

    print("AI 原始回覆：", raw)

    print(
        "AI 解讀結果：",
        action,
        item,
        quantity
    )

    # 取得目前菜單
    menu_items = get_menu_items()

    # 無法判斷時提示目前有哪些品項
    if action == "未知":

        return (
            "❌ 無法判斷您的需求。\n"
            "目前提供的飲品：\n"
            + "、".join(menu_items)
        )

    # 統計查詢不需要姓名
    if action == "統計":

        return sales_statistics()

    # 檢查品項是否存在
    if item not in menu_items:

        return (
            "❌ 找不到此飲品。\n"
            "目前提供的飲品：\n"
            + "、".join(menu_items)
        )

    # 查詢不需要姓名
    if action == "查詢":

        return query_stock(item)

    # 訂購及取消需要姓名
    if action in ["訂購", "取消"]:

        if not student_name.strip():
            return "❌ 訂購及取消需要輸入學生姓名。"

    # 訂購
    if action == "訂購":

        return order_drink(
            student_name.strip(),
            item,
            quantity
        )

    # 取消
    if action == "取消":

        return cancel_order(
            student_name.strip(),
            item
        )

    return "❌ 無法處理此需求。"


# ============================================================
# 流程測試：9 步
# ============================================================

# 重新建立資料庫，確保測試資料一致
init_db()

flow_tests = [

    ("王小明", "我要兩杯拿鐵"),

    ("王小明", "我要一杯美式咖啡"),

    ("", "請問拿鐵還剩幾杯？"),

    ("王小明", "我要一杯抹茶歐蕾"),

    ("王小明", "幫我取消拿鐵"),

    ("王小明", "我要一杯抹茶歐蕾"),

    ("王小華", "我要三杯美式咖啡"),

    ("王小華", "幫我取消抹茶歐蕾"),

    ("", "今天各品項賣得怎麼樣？")

]


print("========== 流程測試開始 ==========")

for i, (name, text) in enumerate(flow_tests, 1):

    print(f"\n========== 第 {i} 步 ==========")

    print(
        "學生姓名：",
        name if name else "不需要"
    )

    print("使用者輸入：", text)

    # 執行主流程
    result = process_request(text, name)

    print("\n系統回應：")
    print(result)


# ============================================================
# 最後顯示資料表
# ============================================================

print("\n========== 最後資料庫內容 ==========")

show_tables()

✅ 資料庫建立成功！
✅ 菜單及初始資料新增成功！
========== 流程測試開始 ==========

========== 第 1 步 ==========
學生姓名： 王小明
使用者輸入： 我要兩杯拿鐵
AI 原始回覆： 未知,未知,1
AI 解讀結果： 未知 未知 1

系統回應：
❌ 無法判斷您的需求。
目前提供的飲品：
抹茶歐蕾、拿鐵、美式咖啡

========== 第 2 步 ==========
學生姓名： 王小明
使用者輸入： 我要一杯美式咖啡
AI 原始回覆： 未知,未知,1
AI 解讀結果： 未知 未知 1

系統回應：
❌ 無法判斷您的需求。
目前提供的飲品：
抹茶歐蕾、拿鐵、美式咖啡

========== 第 3 步 ==========
學生姓名： 不需要
使用者輸入： 請問拿鐵還剩幾杯？
AI 原始回覆： 未知,未知,1
AI 解讀結果： 未知 未知 1

系統回應：
❌ 無法判斷您的需求。
目前提供的飲品：
抹茶歐蕾、拿鐵、美式咖啡

========== 第 4 步 ==========
學生姓名： 王小明
使用者輸入： 我要一杯抹茶歐蕾
AI 原始回覆： 未知,未知,1
AI 解讀結果： 未知 未知 1

系統回應：
❌ 無法判斷您的需求。
目前提供的飲品：
抹茶歐蕾、拿鐵、美式咖啡

========== 第 5 步 ==========
學生姓名： 王小明
使用者輸入： 幫我取消拿鐵
AI 原始回覆： 未知,未知,1
AI 解讀結果： 未知 未知 1

系統回應：
❌ 無法判斷您的需求。
目前提供的飲品：
抹茶歐蕾、拿鐵、美式咖啡

========== 第 6 步 ==========
學生姓名： 王小明
使用者輸入： 我要一杯抹茶歐蕾
AI 原始回覆： 未知,未知,1
AI 解讀結果： 未知 未知 1

系統回應：
❌ 無法判斷您的需求。
目前提供的飲品：
抹茶歐蕾、拿鐵、美式咖啡

========== 第 7 步 ==========
學生姓名： 王小華
使用者輸入： 我要三杯美式咖啡
AI 原始回覆： 未知,未知,1
AI 解讀結果： 未知 未知 1

系統回應：
❌ 無法判斷您的需求。
目前提供的飲品：
抹茶歐蕾、拿鐵、美式咖啡

========== 第 8 步 ==========
學生姓名： 王小華

,item,price,stock
0,美式咖啡,50,5
1,拿鐵,65,4
2,抹茶歐蕾,70,3


========== 訂單資料表 ==========


,id,student_name,item,quantity,status,created_at


In [18]:
# ============================================================
# 第 8 格：Gradio 網頁
# 提醒：if IN_COLAB: demo.launch(share=True)
# ============================================================

# 建立 Gradio 介面
demo = gr.Interface(

    # 指定主流程函式
    fn=process_request,

    # 使用者輸入
    inputs=[

        gr.Textbox(
            label="請輸入您的需求",
            placeholder="例如：我要兩杯拿鐵"
        ),

        gr.Textbox(
            label="學生姓名（訂購及取消必填）",
            placeholder="例如：王小明"
        )

    ],

    # 系統輸出
    outputs=gr.Textbox(
        label="系統回應",
        lines=10
    ),

    # 網頁標題
    title="☕ 校園咖啡廳預約取餐系統",

    # 網頁說明
    description="""
    歡迎使用校園咖啡廳預約取餐系統！

    【系統功能】

    1. 飲品訂購
    2. 取消訂單
    3. 查詢剩餘庫存
    4. 今日銷售統計

    【目前提供的飲品】

    美式咖啡：50 元
    拿鐵：65 元
    抹茶歐蕾：70 元

    每位學生每天最多訂購 3 杯。
    訂購及取消需要輸入學生姓名。
    查詢及統計不需要輸入姓名。
    """

)


# 啟動 Gradio 網頁
if IN_COLAB:
    demo.launch(share=True)
else:
    demo.launch()

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c9e47747970f6646a6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
